# Stage 9 — Debit spread ranking

Full-universe backtest (notebook 14, 214 instruments) showed 59.3% with a
positive avg R but high variance across instruments (std 0.21 vs mean 0.08) -
not a uniform edge. This ranks instruments the same way the other three
strategies are ranked (`strategy_scoring.py`), so the shortlist is the
instruments where this specific signal+lifecycle actually worked, not the
whole universe.

median_r_achieved is weighted highest (not avg_r_achieved) - same
median-over-mean preference as the rest of this project's stats, and it's
what caught BHARTIARTL's outlier-reliant result during validation.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.strategy_scoring import score_for_debit_spread, save_ranking

comparison = pd.read_excel("../exports/DEBIT_SPREAD_COMPARISON.xlsx", sheet_name="Comparison")
comparison.shape

(216, 7)

In [2]:
ranked, watchlist = score_for_debit_spread(comparison)
ranked.head(20)

,symbol,debit_spread_score,outlier_reliance_flag,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held
0,POWERINDIA,98.9,,68,63.2,0.52,1.50,5.0
1,TIINDIA,94.6,,67,53.7,0.27,0.55,5.0
2,CDSL,94.4,,68,57.4,0.41,1.46,6.0
3,PFC,93.3,,62,54.8,0.38,1.23,6.0
4,INFY,93.0,,75,56.0,0.34,0.82,6.0
5,GAIL,92.8,,63,52.4,0.27,0.50,5.0
6,ANGELONE,92.1,,76,53.9,0.33,0.74,6.0
7,BHEL,92.1,,60,56.7,0.32,0.45,6.0
8,SUZLON,91.4,,74,54.1,0.30,0.52,6.0
9,IREDA,90.5,,60,53.3,0.25,0.24,5.5


In [3]:
# How many top-20 candidates carry the outlier-reliance warning?
print(f"Instruments passing min_trades filter: {len(ranked)} of {len(comparison)}")
print(f"Flagged in top 20: {(ranked.head(20)['outlier_reliance_flag'] != '').sum()}")
ranked[ranked["outlier_reliance_flag"] != ""].head(10)

Instruments passing min_trades filter: 214 of 216
Flagged in top 20: 0


,symbol,debit_spread_score,outlier_reliance_flag,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held
32,COFORGE,79.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.32R),73,49.3,0.22,-0.10,5.0
33,JSWENERGY,79.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.33R),64,50.0,0.19,-0.14,5.0
46,ASHOKLEY,73.2,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.36R),71,49.3,0.18,-0.18,6.0
50,DABUR,70.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.36R),61,49.2,0.16,-0.20,6.0
52,RADICO,70.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.31R),70,47.1,0.13,-0.18,5.0
56,SBIN,68.9,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.51R),65,49.2,0.19,-0.32,6.0
57,FORCEMOT,68.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.47R),66,48.5,0.19,-0.28,6.0
59,PNBHOUSING,68.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.37R),62,48.4,0.16,-0.21,6.0
61,GODREJPROP,66.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.47R),62,48.4,0.17,-0.30,6.0
62,SAIL,66.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.67R),61,49.2,0.20,-0.47,6.0


In [4]:
save_ranking(ranked, "RANKING_debit_spread.xlsx")

Saved ranking to /Users/hgmac/Documents/3_trading/Quant/exports/RANKING_debit_spread.xlsx


'/Users/hgmac/Documents/3_trading/Quant/exports/RANKING_debit_spread.xlsx'

In [5]:
# Instruments below min_trades - too thin a sample to blend into the
# percentile ranking above, but not dropped either (e.g. SWIGGY: recently
# added to F&O, so it hasn't had time to generate enough signals yet).
# Sorted by avg_r_achieved so a promising-but-unproven name is visible,
# not silently missing. Re-check at the next periodic refresh - trade
# count grows on its own as more calendar time passes.
print(f"Instruments on watchlist (below min_trades): {len(watchlist)}")
watchlist

Instruments on watchlist (below min_trades): 2


,symbol,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held
0,SWIGGY,29,65.5,0.54,1.5,6.0
1,ATHERENERG,24,62.5,0.50,1.5,5.0


In [6]:
save_ranking(watchlist, "WATCHLIST_debit_spread.xlsx")

Saved ranking to /Users/hgmac/Documents/3_trading/Quant/exports/WATCHLIST_debit_spread.xlsx


'/Users/hgmac/Documents/3_trading/Quant/exports/WATCHLIST_debit_spread.xlsx'